# 05 - Inference Testing (Google Colab)
Load base model + LoRA adapter, run inference on sample resume + job description, validate JSON output.

In [ ]:
# Mount Google Drive and set working directory
from google.colab import drive
drive.mount('/content/drive')

import os
COLAB_ROOT = '/content/drive/MyDrive/colab'
os.chdir(COLAB_ROOT)
print(f"Working directory: {os.getcwd()}")

In [ ]:
import json
import re
from pathlib import Path

import yaml
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

In [ ]:
# Load config
with open("configs/training_config.yaml", "r") as f:
    config = yaml.safe_load(f)

model_name = config["model_name"]
adapter_path = config["output_dir"]
use_4bit = config.get("use_4bit", True)

print(f"Base model: {model_name}")
print(f"Adapter path: {adapter_path}")
print(f"4-bit quantization: {use_4bit}")

In [ ]:
# Step 1: Load base model + LoRA adapter
print(f"Loading base model: {model_name}")

# Quantization
bnb_config = None
if use_4bit and torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

# Tokenizer
try:
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
except Exception:
    model_name = config["fallback_model"]
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Model
model_kwargs = {
    "trust_remote_code": True,
    "torch_dtype": torch.float16,
    "device_map": "auto",
}
if bnb_config:
    model_kwargs["quantization_config"] = bnb_config

model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)

# Load LoRA adapter
print(f"Loading LoRA adapter from: {adapter_path}")
model = PeftModel.from_pretrained(model, adapter_path)
model.eval()

print("Model loaded successfully!")

In [ ]:
# Step 2: Define helper functions
def format_prompt(instruction, resume_text, job_description):
    """Format the inference prompt."""
    input_text = f"RESUME:\n{resume_text}\n\nJOB DESCRIPTION:\n{job_description}"
    return (
        f"### Instruction:\n{instruction}\n\n"
        f"### Input:\n{input_text}\n\n"
        f"### Response:\n"
    )

def extract_json(text):
    """Extract JSON from model output."""
    json_match = re.search(r'\{[\s\S]*\}', text)
    if json_match:
        try:
            return json.loads(json_match.group())
        except json.JSONDecodeError:
            pass
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

def validate_ats_output(output):
    """Validate ATS JSON has required fields."""
    required = {"ats_score", "score_breakdown", "matched_skills",
                "missing_skills", "weak_bullets", "formatting_issues", "overall_feedback"}
    if not required.issubset(output.keys()):
        return False
    if not isinstance(output["ats_score"], (int, float)) or not (0 <= output["ats_score"] <= 100):
        return False
    return True

def generate_ats_eval(resume_text, job_description, max_new_tokens=1024, temperature=0.1):
    """Generate ATS evaluation."""
    instruction = (
        "Evaluate the following resume against the job description and provide a detailed "
        "ATS (Applicant Tracking System) compliance analysis. Return a structured JSON evaluation "
        "including ATS score, score breakdown, matched skills, missing skills, weak bullet analysis "
        "with improvements, formatting issues, and overall feedback."
    )
    prompt = format_prompt(instruction, resume_text, job_description)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=0.9,
            repetition_penalty=1.1,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    result = extract_json(generated_text)
    if result and validate_ats_output(result):
        result["valid_json"] = True
    else:
        result = {"raw_output": generated_text, "valid_json": False}

    return result

print("Helper functions defined")

In [ ]:
# Step 3: Test with sample resume
sample_resume = """John Smith
Software Engineer | john.smith@email.com | (555) 123-4567

EXPERIENCE
Software Engineer, TechCorp Inc. - Jan 2021 - Present
- Developed RESTful APIs using Python and Flask serving 10K daily users
- Managed PostgreSQL databases with 50+ tables and optimized query performance
- Collaborated with cross-functional teams to deliver 3 major product releases
- Responsible for maintaining CI/CD pipelines using Jenkins and Docker

Junior Developer, StartupXYZ - Jun 2019 - Dec 2020
- Built frontend components using React and TypeScript
- Helped with bug fixes and code reviews
- Participated in daily standup meetings and sprint planning

EDUCATION
B.S. Computer Science, State University - 2019

SKILLS
Python, JavaScript, React, Flask, PostgreSQL, Docker, Git, AWS"""

sample_jd = """Senior Software Engineer
TechGlobal Inc.

Requirements:
- 5+ years of experience in software development
- Strong proficiency in Python and Java
- Experience with RESTful API design and microservices architecture
- Proficiency with SQL databases (PostgreSQL preferred)
- Experience with cloud services (AWS or GCP)
- Familiarity with Docker and Kubernetes
- Strong understanding of CI/CD pipelines

Responsibilities:
- Design and implement scalable backend services
- Write clean, maintainable, and well-tested code
- Participate in code reviews and technical design discussions
- Mentor junior engineers"""

print("Running inference...")
result = generate_ats_eval(sample_resume, sample_jd)

print("\n--- ATS Evaluation Result ---\n")
print(json.dumps(result, indent=2))

In [ ]:
# Step 4: Validate output quality
if result.get("valid_json"):
    print("JSON Validation: PASSED")
    print(f"\nATS Score: {result['ats_score']}/100")

    print(f"\nScore Breakdown:")
    for key, val in result["score_breakdown"].items():
        print(f"  {key}: {val}")

    print(f"\nMatched Skills ({len(result['matched_skills'])}):")
    for skill in result["matched_skills"]:
        print(f"  + {skill}")

    print(f"\nMissing Skills ({len(result['missing_skills'])}):")
    for skill in result["missing_skills"]:
        print(f"  - {skill}")

    print(f"\nWeak Bullets ({len(result['weak_bullets'])}):")
    for wb in result["weak_bullets"]:
        print(f"  Original: {wb['original']}")
        print(f"  Issue: {wb['issue']}")
        print(f"  Improved: {wb['improved']}")
        print()

    print(f"Formatting Issues: {len(result['formatting_issues'])}")
    print(f"Overall Feedback: {result['overall_feedback'][:200]}...")
else:
    print("JSON Validation: FAILED")
    print(f"Raw output: {result.get('raw_output', 'N/A')[:500]}")

In [ ]:
# Step 5: Test with a different resume/JD pair
sample_resume_2 = """Sarah Chen
Data Scientist | sarah.chen@email.com

EXPERIENCE
Senior Data Scientist, DataDriven Corp - Mar 2022 - Present
- Built machine learning models achieving 92% accuracy on customer churn prediction
- Designed A/B testing framework that increased conversion rates by 15%
- Processed and analyzed datasets containing 10M+ records using PySpark

Data Analyst, Analytics Co - Aug 2020 - Feb 2022
- Created dashboards using Tableau
- Responsible for data cleaning tasks
- Assisted in building predictive models

EDUCATION
M.S. Statistics, Top University - 2020

SKILLS
Python, R, SQL, TensorFlow, PyTorch, Tableau, PySpark, scikit-learn"""

sample_jd_2 = """Machine Learning Engineer
AIVentures Inc.

Requirements:
- 3+ years of ML engineering experience
- Strong Python skills
- Experience with PyTorch or TensorFlow
- Experience deploying ML models to production
- Knowledge of MLOps (MLflow, Kubeflow)
- Experience with NLP or Computer Vision

Responsibilities:
- Design and implement ML pipelines
- Deploy and monitor models in production
- Optimize model performance"""

print("Running inference on second sample...")
result_2 = generate_ats_eval(sample_resume_2, sample_jd_2)

print("\n--- ATS Evaluation Result (Sample 2) ---\n")
print(json.dumps(result_2, indent=2))

print(f"\nValid JSON: {result_2.get('valid_json', False)}")
if result_2.get('valid_json'):
    print(f"ATS Score: {result_2['ats_score']}/100")

In [ ]:
# Step 6: Test determinism (same input should give similar output)
print("Testing output consistency (temperature=0.0)...")

result_det1 = generate_ats_eval(sample_resume, sample_jd, temperature=0.0)
result_det2 = generate_ats_eval(sample_resume, sample_jd, temperature=0.0)

if result_det1.get("valid_json") and result_det2.get("valid_json"):
    score_match = result_det1["ats_score"] == result_det2["ats_score"]
    skills_match = result_det1["matched_skills"] == result_det2["matched_skills"]
    print(f"Score match: {score_match} ({result_det1['ats_score']} vs {result_det2['ats_score']})")
    print(f"Skills match: {skills_match}")
    print(f"Deterministic: {'YES' if score_match and skills_match else 'PARTIAL'}")
else:
    print("Cannot test determinism - invalid JSON output")